# PARTE A

In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('spanish'))

df = pd.read_csv('../data/libros.csv')

# Descartamos filas sin sinopsis
df = df.dropna(subset=['sinopsis']).reset_index(drop=True)

def limpiar_crudo(texto):
    return re.sub(r'\s+', ' ', str(texto)).strip()

def limpiar_tokenizado(texto):
    texto = str(texto).lower()
    texto = re.sub(r'[^\w\s]', ' ', texto)
    texto = re.sub(r'\d+', ' ', texto)
    
    palabras = texto.split()
    palabras_limpias = [p for p in palabras if p not in stop_words]
    
    return ' '.join(palabras_limpias)

df['sinopsis_cruda'] = df['sinopsis'].apply(limpiar_crudo)
df['sinopsis_limpia'] = df['sinopsis'].apply(limpiar_tokenizado)

print("Muestra del texto crudo (con contexto):")
print(df['sinopsis_cruda'].iloc[0][:150], "...\n")

print("Muestra del texto limpio (bolsa de palabras):")
print(df['sinopsis_limpia'].iloc[0][:150], "...")

# PARTE B

In [ ]:
from gensim.models import Word2Vec, KeyedVectors
import gdown
import os

# 1. Preparamos el corpus para tu modelo (lista de listas de palabras)
oraciones_propias = [texto.split() for texto in df['sinopsis_limpia']]

# 2. Entrenamos tu propio Word2Vec
# Parámetros a justificar en tu informe:
# - vector_size=100: Como tu corpus es diminuto (150 libros), 300 dimensiones generaría vectores muy ruidosos.
# - window=5: Ventana de contexto estándar (5 palabras a la izquierda, 5 a la derecha).
# - min_count=2: Descartamos palabras que aparecen 1 sola vez (hapax) para no generar ruido.
# - sg=1: Usamos Skip-gram, que suele funcionar mejor que CBOW con datasets pequeños.
print("Entrenando modelo Word2Vec propio...")
modelo_propio = Word2Vec(sentences=oraciones_propias, vector_size=100, window=5, min_count=2, sg=1)

# 3. Descargamos y cargamos el modelo pre-entrenado (Spanish Billion Words)
ruta_modelo_sbw = '../data/SBW-vectors-300-min5.bin.gz'

if not os.path.exists(ruta_modelo_sbw):
    print("\nDescargando el modelo pre-entrenado SBW (esto puede tardar unos minutos)...")
    url_sbw = 'https://drive.google.com/uc?id=1V8hNcnGEyrz0c_dA-v0_5sg31bNgy75r'
    gdown.download(url_sbw, ruta_modelo_sbw, quiet=False)

print("\nCargando el modelo pre-entrenado (requiere bastante RAM)...")
modelo_sbw = KeyedVectors.load_word2vec_format(ruta_modelo_sbw, binary=True)

# 4. Comparamos los vecinos más cercanos de 4 palabras del dominio "Clásicos"
palabras_dominio = ['amor', 'guerra', 'muerte', 'familia']

print("\n--- COMPARACIÓN DE VECINOS MÁS CERCANOS ---")
for palabra in palabras_dominio:
    print(f"\nPalabra objetivo: '{palabra.upper()}'")
    
    # Vecinos en tu modelo
    print("Vecinos en tu modelo propio:")
    if palabra in modelo_propio.wv:
        vecinos_propios = modelo_propio.wv.most_similar(palabra, topn=3)
        for v, sim in vecinos_propios:
            print(f"  - {v} ({sim:.3f})")
    else:
        print("  (La palabra no existe en el vocabulario de tu corpus)")
        
    # Vecinos en el modelo gigante
    print("Vecinos en modelo pre-entrenado (SBW):")
    if palabra in modelo_sbw:
        vecinos_sbw = modelo_sbw.most_similar(palabra, topn=3)
        for v, sim in vecinos_sbw:
            print(f"  - {v} ({sim:.3f})")
    else:
        print("  (La palabra no existe en SBW)")

# PARTE C

In [ ]:
from sentence_transformers import SentenceTransformer

# 1. Cargamos el modelo pre-entrenado indicado en la cátedra
nombre_modelo = 'distiluse-base-multilingual-cased-v1'
print(f"Cargando modelo {nombre_modelo}...")
modelo_sbert = SentenceTransformer(nombre_modelo)

# 2. Reportamos la dimensión y el límite de tokens
dimension = modelo_sbert.get_sentence_embedding_dimension()
limite_tokens = modelo_sbert.max_seq_length

print("\n--- REPORTES DEL MODELO ---")
print(f"Dimensión de los embeddings: {dimension}")
print(f"Límite de tokens del modelo: {limite_tokens}")

# 3. Calculamos cuántos documentos se truncan
# Usamos el tokenizador interno del modelo para contar los tokens reales de cada sinopsis cruda
tokenizer = modelo_sbert.tokenizer
truncados = 0

for texto in df['sinopsis_cruda']:
    # Tokenizamos el texto sin truncarlo para ver su longitud real
    tokens = tokenizer.encode(texto, add_special_tokens=True)
    if len(tokens) > limite_tokens:
        truncados += 1
        
print(f"Documentos truncados por superar el límite: {truncados} de {len(df)}")

# 4. Generamos los embeddings para todo el corpus
print("\nGenerando embeddings de documentos (esto puede tardar unos segundos)...")
# ATENCIÓN: Le pasamos la sinopsis CRUDA (con puntuación y stopwords)
embeddings_sbert = modelo_sbert.encode(df['sinopsis_cruda'].tolist(), show_progress_bar=True)

print("Embeddings generados exitosamente. Forma de la matriz:", embeddings_sbert.shape)